As part of your analysis, you'll conduct hypothesis testing to make data-driven conclusions about the effectiveness of the redesign. See the full details below:

Completion Rate
Completion Rate with a Cost-Effectiveness Threshold
Other Hypothesis Examples
Experiment Evaluation

You are also required to carry out an evaluation of the experiment by answering questions about the design effectiveness, duration and any additional data needs. See the full details below:

Design Effectiveness
Duration Assessment
Was the timeframe of the experiment (from 3/15/2017 to 6/20/2017) adequate to gather meaningful data and insights?

# 1 analysis de la efectividad del rediseño.

Usaremos df_experiment para separar a los usuarios en Test (rediseño) y Control (diseño viejo) y lo cruzaremss con df_web para ver quién completó el proceso.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data_raw"

print("Directorio actual:", ROOT)
print("Ruta de datos:", DATA_PATH)

Directorio actual: C:\Users\naila\OneDrive\Documentos\Nueva carpeta (2)\Projecto-2-Vanguard-ab-test
Ruta de datos: C:\Users\naila\OneDrive\Documentos\Nueva carpeta (2)\Projecto-2-Vanguard-ab-test\data_raw


In [5]:
# Guardar datasets limpios
df_demo.to_csv(DATA_PATH / "df_demo_clean.csv", index=False)
df_experiment.to_csv(DATA_PATH / "df_experiment_clean.csv", index=False)
df_web.to_csv(DATA_PATH / "df_web_clean.csv", index=False)

print("Datasets guardados correctamente!")
print("df_demo_clean:      ", len(df_demo), "filas")
print("df_experiment_clean:", len(df_experiment), "filas")
print("df_web_clean:       ", len(df_web), "filas")

NameError: name 'df_demo' is not defined

In [ ]:
# Cargar datos limpios
import pandas as pd

def inspect_datasets(dataframes: list[pd.DataFrame], names: list[str]) -> None:
    """
    Imprime las columnas de cada dataset para verificar su estructura.
    """
    for df, name in zip(dataframes, names):
        print(f"\n--- Columnas en {name} ---")
        print(list(df.columns))

# Lista de tus DataFrames y sus nombres para el bucle
datasets = [df_demo, df_experiment, df_web]
dataset_names = ["df_demo", "df_experiment", "df_web"]

# Ejecutar la inspección
inspect_datasets(datasets, dataset_names)



NameError: name 'df_demo' is not defined

In [ ]:
df.columns 

A. Completion Rate (Tasa de Finalización)

Hipótesis Nula (H_0): No hay diferencia en la tasa de finalización entre el grupo de Test y el de Control.

Hipótesis Alternativa (H_1): El nuevo diseño (Test) tiene una tasa de finalización significativamente mayor.

Método: Realizaremos un test para proporciones. Si el p-value es menor a 0.05, rechazamos H_0 y confirmamos que el rediseño es efectivo.

In [ ]:
# hpotisis testing: Tasa de Finalización
# Identificamos quién llegó al paso 'confirm'
df_web['is_completed'] = (df_web['process_step'] == 'confirm').astype(int)

# Agrupamos por cliente (1 si completó al menos una vez, 0 si nunca)
df_conversion = df_web.groupby('client_id')['is_completed'].max().reset_index()

# Unimos con el experimento
df_test_logic = pd.merge(df_conversion, df_experiment, on='client_id')

# Separamos los grupos
group_control = df_test_logic[df_test_logic['Variation'] == 'Control']['is_completed']
group_test = df_test_logic[df_test_logic['Variation'] == 'Test']['is_completed']

In [6]:
# Usamos el T-test (equal_var=False) por ser más robusto 

t_stat, p_value = stats.ttest_ind(group_test, group_control, equal_var=False)
print(f"Tasa de Finalización - Test: {group_test.mean():.2%}")
print(f"Tasa de Finalización - Control: {group_control.mean():.2%}")
print(f"P-value: {p_value:.4f}")

# Interpretación
if p_value < 0.05:
    print("La diferencia en las tasas de finalización es estadísticamente significativa.")
else:
    print("No se observa una diferencia significativa entre los diseños.")

Tasa de Finalización - Test: 69.29%
Tasa de Finalización - Control: 65.59%
P-value: 0.0000
La diferencia en las tasas de finalización es estadísticamente significativa.


Conclusiones
- Design Effectiveness: teniendo en cuenta que el p-value es menor a 0.05 (0.000), podemos concluir que el rediseño tiene un impacto real en el comportamiento del usuario.
- Completion Rate: la tasa de finalización ha aumentado de 3,7 puntos.
- Teniendo en cuenta que tu experimento abarca desde marzo hasta junio de 2017, este T-test es muy robusto porque el tamaño de la muestra acumulado en esos meses reduce significativamente el error estándar.


B. Cost-Effectiveness Threshold

Aquí no solo buscamos que sea "mejor", sino que supere un umbral (ej. un aumento del 5%.
Análisis: Si el coste de implementar el rediseño es alto, la mejora observada debe justificar la inversión.

In [ ]:
# hpotisis testing: Cost-Effectiveness Threshold
# Definimos el umbral (5%)

threshold = 0.05

# Calculamos las tasas de finalización (medias)
rate_control = group_control.mean()
rate_test = group_test.mean()

# Calculamos la mejora absoluta
observed_diff = rate_test - rate_control

print(f"Mejora observada: {observed_diff:.2%}")

Mejora observada: 3.71%


In [8]:
# 1. Definir la métrica de éxito (Completion)
# 1 si el cliente llegó a 'confirm', 0 en caso contrario
df_web['is_completed'] = (df_web['process_step'] == 'confirm').astype(int)

# 2. Agrupar por cliente para obtener un valor único por persona
# Si un cliente tiene al menos un 1, su máximo será 1 (completado)
df_conversion = df_web.groupby('client_id')['is_completed'].max().reset_index()

# 3. Unir con el experimento para obtener los grupos
df_merged = pd.merge(df_conversion, df_experiment, on='client_id')

# 4. Separar los datos por variación
# Seleccionamos solo la columna 'is_completed' para cada grupo
group_test = df_merged[df_merged['Variation'] == 'Test']['is_completed']
group_control = df_merged[df_merged['Variation'] == 'Control']['is_completed']

# 5. Ejecutar el T-test de muestras independientes
# Usamos equal_var=False para aplicar el test de Welch, que es más seguro
t_stat, p_value = stats.ttest_ind(group_test, group_control, equal_var=False)

# Mostrar resultados formateados
print(f"Tasa de Finalización (Test): {group_test.mean():.2%}")
print(f"Tasa de Finalización (Control): {group_control.mean():.2%}")
print(f"P-value: {p_value:.4f}")

Tasa de Finalización (Test): 69.29%
Tasa de Finalización (Control): 65.59%
P-value: 0.0000


C. Edad y Tenencia 

Dado que tenemos df_demo, podemos verificar si el rediseño funciona mejor para ciertos segmentos:

¿Influye la edad (clnt_age) en la probabilidad de completar el proceso?

¿Los clientes más antiguos (clnt_tenure_yr) son más resistentes al cambio que los nuevos?

In [9]:
# hpotisis testing: Edad y Tenencia 
# Unimos los datos de conversión (que ya calculamos) con df_demo y df_experiment
df_segmented = pd.merge(df_conversion, df_demo, on='client_id')
df_segmented = pd.merge(df_segmented, df_experiment, on='client_id')

# Filtramos por los grupos de interés
test_group = df_segmented[df_segmented['Variation'] == 'Test']
control_group = df_segmented[df_segmented['Variation'] == 'Control']

In [10]:
# Comparamos la edad de los que finalizaron vs los que no en el grupo Test
completed_age = test_group[test_group['is_completed'] == 1]['clnt_age']
not_completed_age = test_group[test_group['is_completed'] == 0]['clnt_age']

# T-test para ver si hay una diferencia significativa de edad en el éxito
t_stat_age, p_val_age = stats.ttest_ind(completed_age, not_completed_age, equal_var=False)

print(f"Edad media (Completado): {completed_age.mean():.1f}")
print(f"Edad media (No Completado): {not_completed_age.mean():.1f}")
print(f"P-value edad: {p_val_age:.4f}")

Edad media (Completado): 46.6
Edad media (No Completado): 48.5
P-value edad: 0.0000


In [12]:
# Analizamos la antigüedad en el grupo que falló en el rediseño (Test) 
# frente al grupo que falló en el diseño viejo (Control)
tenure_fail_test = test_group[test_group['is_completed'] == 0]['clnt_tenure_yr']
tenure_fail_control = control_group[control_group['is_completed'] == 0]['clnt_tenure_yr']

t_stat_tenure, p_val_tenure = stats.ttest_ind(tenure_fail_test, tenure_fail_control, equal_var=False)

print(f"Antigüedad media en abandonos (Test): {tenure_fail_test.mean():.1f} años")
print(f"Antigüedad media en abandonos (Control): {tenure_fail_control.mean():.1f} años")

Antigüedad media en abandonos (Test): 12.1 años
Antigüedad media en abandonos (Control): 12.2 años
